# SHA Phase 1151 — Reference Trace Lock + Formula Firewall v1

## Fix the schedule before descent

This notebook implements the gate:

$$
\boxed{\text{Reference Trace Lock before Need descent.}}
$$

If the correct preimage does not produce zero digest cost, then the constraint surface is corrupted.

This notebook locks:

$$
H_{\text{engine}}(M)=H_{\text{hashlib}}(M)
$$

for multiple single-block messages.

It also installs a **formula firewall**:

- Sziklai is a structural floor, not a filter.
- Rank/Free Filter is not assumed; it must be extracted from a concrete Jacobian projection.
- Ghost/Shape projection is measured, not asserted.
- Need descent is not run until the true message has zero digest cost.

Core schedule:

$$
W_t=\sigma_1(W_{t-2})+W_{t-7}+\sigma_0(W_{t-15})+W_{t-16}\pmod{2^{32}}.
$$

Core round:

$$
T1_r=h_r+\Sigma_1(e_r)+Ch(e_r,f_r,g_r)+K_r+W_r
$$

$$
T2_r=\Sigma_0(a_r)+Maj(a_r,b_r,c_r)
$$

$$
a_{r+1}=T1_r+T2_r,\qquad e_{r+1}=d_r+T1_r.
$$

Sziklai floor:

$$
a_{r+1}-e_{r+1}\equiv T2_r-d_r\pmod{2^{32}}.
$$


In [1]:
# Imports and settings

import hashlib
import random
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MASK = 0xFFFFFFFF
SEED = 1151
random.seed(SEED)
np.random.seed(SEED)

print("Phase 1151 Reference Trace Lock initialized.")


Phase 1151 Reference Trace Lock initialized.


In [2]:
# SHA-256 constants

H0 = [
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
    0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19,
]

K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

print("Constants loaded:", len(H0), len(K))


Constants loaded: 8 64


## 1. Word primitives

In [3]:
# Word primitives

def u32(x: int) -> int:
    return x & MASK

def rotr(x: int, n: int) -> int:
    return ((x >> n) | ((x << (32 - n)) & MASK)) & MASK

def shr(x: int, n: int) -> int:
    return (x >> n) & MASK

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

def Ch(x: int, y: int, z: int) -> int:
    return (x & y) ^ ((~x) & z)

def Maj(x: int, y: int, z: int) -> int:
    return (x & y) ^ (x & z) ^ (y & z)

def add_mod(*xs: int) -> int:
    return sum(xs) & MASK

def xor_face(*xs: int) -> int:
    y = 0
    for x in xs:
        y ^= x & MASK
    return y & MASK

def scar_for_add(*xs: int) -> int:
    return add_mod(*xs) ^ xor_face(*xs)

def hw_bytes(bs: bytes) -> int:
    return sum(int(b).bit_count() for b in bs)

def fmtw(x: int) -> str:
    return f"{x & MASK:08x}"

print("Primitives loaded.")


Primitives loaded.


## 2. Single-block padding and schedule

For a one-block message:

$$
|M|\le55
$$

the final 64 bits are the message length in **bits**, big-endian.


In [4]:
# Padding and schedule

def pad_single_block(msg: bytes) -> bytes:
    if len(msg) > 55:
        raise ValueError("single-block only: len(msg) <= 55")
    bit_len = len(msg) * 8
    block = bytearray(msg)
    block.append(0x80)
    while len(block) != 56:
        block.append(0x00)
    block += bit_len.to_bytes(8, "big")
    assert len(block) == 64
    return bytes(block)

def block_words_be(block: bytes) -> List[int]:
    assert len(block) == 64
    return [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]

def expand_schedule_from_words(W16: List[int]) -> List[int]:
    W = list(W16)
    assert len(W) == 16
    for t in range(16, 64):
        W.append(add_mod(sigma1(W[t-2]), W[t-7], sigma0(W[t-15]), W[t-16]))
    return W

def schedule_from_msg(msg: bytes) -> List[int]:
    return expand_schedule_from_words(block_words_be(pad_single_block(msg)))

for msg in [b"", b"abc", bytes(range(32)), bytes(range(55))]:
    block = pad_single_block(msg)
    assert len(block) == 64
    assert int.from_bytes(block[-8:], "big") == len(msg) * 8

print("Padding and schedule loaded.")
print("abc W[0..3]:", [fmtw(w) for w in schedule_from_msg(b"abc")[:4]])
print("abc W[15]:", fmtw(schedule_from_msg(b"abc")[15]))


Padding and schedule loaded.
abc W[0..3]: ['61626380', '00000000', '00000000', '00000000']
abc W[15]: 00000018


## 3. Reference compression trace

In [5]:
# Compression trace

def sha256_trace_one_block(msg: bytes) -> Tuple[List[int], List[Dict[str, int]], bytes]:
    W = schedule_from_msg(msg)
    a,b,c,d,e,f,g,h = H0
    rows = []

    for r in range(64):
        S1 = Sigma1(e)
        ch = Ch(e, f, g)
        S0 = Sigma0(a)
        maj = Maj(a, b, c)

        T1 = add_mod(h, S1, ch, K[r], W[r])
        T2 = add_mod(S0, maj)

        a_next = add_mod(T1, T2)
        e_next = add_mod(d, T1)

        rows.append({
            "r": r,
            "a": a, "b": b, "c": c, "d": d,
            "e": e, "f": f, "g": g, "h": h,
            "W": W[r],
            "Sigma1e": S1,
            "Ch": ch,
            "Sigma0a": S0,
            "Maj": maj,
            "T1": T1,
            "T2": T2,
            "a_next": a_next,
            "e_next": e_next,
            "scar_T1": scar_for_add(h, S1, ch, K[r], W[r]),
            "scar_T2": scar_for_add(S0, maj),
            "scar_a": scar_for_add(T1, T2),
            "scar_e": scar_for_add(d, T1),
            "sziklai_deviation": u32((a_next - e_next) - (T2 - d)),
        })

        a,b,c,d,e,f,g,h = a_next,a,b,c,e_next,e,f,g

    digest_words = [add_mod(x, y) for x, y in zip([a,b,c,d,e,f,g,h], H0)]
    digest = b"".join(w.to_bytes(4, "big") for w in digest_words)
    return W, rows, digest

def engine_sha256(msg: bytes) -> bytes:
    return sha256_trace_one_block(msg)[2]

print("Trace function loaded.")


Trace function loaded.


## 4. Reference Trace Lock

Hard gate:

$$
H_{\text{engine}}(M)=H_{\text{hashlib}}(M).
$$


In [6]:
# Reference Trace Lock test vectors

test_vectors = {
    "empty": b"",
    "abc": b"abc",
    "32_bytes_range": bytes(range(32)),
    "55_bytes_range": bytes(range(55)),
    "32_bytes_ff": bytes([0xFF]) * 32,
    "32_bytes_zero": bytes([0x00]) * 32,
}

lock_rows = []
for name, msg in test_vectors.items():
    W, rows, digest = sha256_trace_one_block(msg)
    ref = hashlib.sha256(msg).digest()
    match = digest == ref
    sz_ok = all(row["sziklai_deviation"] == 0 for row in rows)
    lock_rows.append({
        "name": name,
        "len_bytes": len(msg),
        "engine_digest": digest.hex(),
        "hashlib_digest": ref.hex(),
        "digest_match": match,
        "sziklai_all_zero": sz_ok,
        "W15": fmtw(W[15]),
        "W16": fmtw(W[16]),
        "W63": fmtw(W[63]),
    })
    assert match, f"digest mismatch for {name}"
    assert sz_ok, f"Sziklai deviation for {name}"

lock_df = pd.DataFrame(lock_rows)
lock_df


,name,len_bytes,engine_digest,hashlib_digest,digest_match,sziklai_all_zero,W15,W16,W63
0,empty,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True,True,00000000,80000000,3b5ec49b
1,abc,3,ba7816bf8f01cfea414140de5dae2223b00361a396177a...,ba7816bf8f01cfea414140de5dae2223b00361a396177a...,True,True,00000018,61626380,12b1edeb
2,32_bytes_range,32,630dcd2966c4336691125448bbb25b4ff412a49c732db2...,630dcd2966c4336691125448bbb25b4ff412a49c732db2...,True,True,00000100,4f0a6dd0,b29dedff
3,55_bytes_range,55,463eb28e72f82e0a96c0a4cc53690c571281131f672aa2...,463eb28e72f82e0a96c0a4cc53690c571281131f672aa2...,True,True,000001b8,732f93f7,17395ecf
4,32_bytes_ff,32,af9613760f72635fbdb44a5a0a63c39f12af30f950a6ee...,af9613760f72635fbdb44a5a0a63c39f12af30f950a6ee...,True,True,00000100,1ffffffe,ddff2275
5,32_bytes_zero,32,66687aadf862bd776c8fc18b8e9f8e20089714856ee233...,66687aadf862bd776c8fc18b8e9f8e20089714856ee233...,True,True,00000100,00000000,25c9488c


## 5. Known-preimage zero-cost check

For target digest $\sigma=H(M_{\text{true}})$:

$$
C_{\text{digest}}(M_{\text{true}})=0.
$$


In [7]:
# Digest cost

def digest_cost_bits(candidate: bytes, target_digest: bytes) -> int:
    return hw_bytes(bytes(a ^ b for a, b in zip(engine_sha256(candidate), target_digest)))

cost_rows = []
for name, msg in test_vectors.items():
    target = hashlib.sha256(msg).digest()
    true_cost = digest_cost_bits(msg, target)

    if len(msg) > 0:
        perturbed = bytearray(msg)
        perturbed[0] ^= 0x80
        perturbed = bytes(perturbed)
        perturbed_cost = digest_cost_bits(perturbed, target)
    else:
        perturbed_cost = None

    cost_rows.append({
        "name": name,
        "true_cost": true_cost,
        "one_bit_perturbed_cost": perturbed_cost,
    })
    assert true_cost == 0

cost_df = pd.DataFrame(cost_rows)
cost_df


,name,true_cost,one_bit_perturbed_cost
0,empty,0,NaN
1,abc,0,128.0
2,32_bytes_range,0,128.0
3,55_bytes_range,0,133.0
4,32_bytes_ff,0,125.0
5,32_bytes_zero,0,125.0


## 6. Formula Firewall

In [8]:
# Formula firewall status table

firewall = pd.DataFrame([
    {
        "claim": "SHA round recurrence",
        "status": "LOCKED",
        "code_gate": "engine digest equals hashlib for all test vectors",
    },
    {
        "claim": "Sziklai relation a'-e' = T2-d",
        "status": "LOCKED",
        "code_gate": "deviation zero across all 64 rounds",
    },
    {
        "claim": "CSA/carry scar extraction",
        "status": "LOCAL-LOCKED",
        "code_gate": "scar_for_add = ADD xor XOR face by construction",
    },
    {
        "claim": "Rank-4 Free Filter",
        "status": "PROJECTION-DEPENDENT",
        "code_gate": "must extract ker(J.T) from concrete CSA/seam map",
    },
    {
        "claim": "160 visible ghost bits",
        "status": "UNRESOLVED",
        "code_gate": "must appear as stable digest/ghost projection dimension",
    },
    {
        "claim": "Need descent recovers preimage",
        "status": "NOT YET",
        "code_gate": "only after reference lock and controlled-subspace demo",
    },
    {
        "claim": "matter replication / ZPHC physical generation",
        "status": "OMEGA",
        "code_gate": "outside this SHA executable notebook",
    },
])

firewall


,claim,status,code_gate
0,SHA round recurrence,LOCKED,engine digest equals hashlib for all test vectors
1,Sziklai relation a'-e' = T2-d,LOCKED,deviation zero across all 64 rounds
2,CSA/carry scar extraction,LOCAL-LOCKED,scar_for_add = ADD xor XOR face by construction
3,Rank-4 Free Filter,PROJECTION-DEPENDENT,must extract ker(J.T) from concrete CSA/seam map
4,160 visible ghost bits,UNRESOLVED,must appear as stable digest/ghost projection ...
5,Need descent recovers preimage,NOT YET,only after reference lock and controlled-subsp...
6,matter replication / ZPHC physical generation,OMEGA,outside this SHA executable notebook


## 7. Ghost vector extraction after trace lock

Ghost definition:

$$
G(M)=
[S_{T1,r},S_{T2,r},S_{a,r},S_{e,r}]_{r=58}^{63}
$$

This is:

$$
6\times4\times32=768
$$

bits.


In [9]:
# Ghost vector helpers

GHOST_ROUNDS = list(range(58, 64))

def words_to_bits_be(words: List[int]) -> np.ndarray:
    bits = []
    for w in words:
        for i in range(31, -1, -1):
            bits.append((w >> i) & 1)
    return np.array(bits, dtype=np.uint8)

def ghost_words(msg: bytes, ghost_rounds=GHOST_ROUNDS) -> List[int]:
    _, rows, _ = sha256_trace_one_block(msg)
    out = []
    for r in ghost_rounds:
        row = rows[r]
        out += [row["scar_T1"], row["scar_T2"], row["scar_a"], row["scar_e"]]
    return out

def ghost_bits(msg: bytes) -> np.ndarray:
    return words_to_bits_be(ghost_words(msg))

ghost_audit_rows = []
for name, msg in test_vectors.items():
    g = ghost_bits(msg)
    ghost_audit_rows.append({
        "name": name,
        "ghost_bits": len(g),
        "ghost_hw": int(g.sum()),
        "ghost_density": float(g.mean()),
    })

ghost_audit_df = pd.DataFrame(ghost_audit_rows)
ghost_audit_df


,name,ghost_bits,ghost_hw,ghost_density
0,empty,768,311,0.404948
1,abc,768,371,0.483073
2,32_bytes_range,768,397,0.516927
3,55_bytes_range,768,397,0.516927
4,32_bytes_ff,768,356,0.463542
5,32_bytes_zero,768,334,0.434896


## 8. Schedule diagnostics table

In [10]:
# Schedule diagnostics table

diag_rows = []
for name, msg in test_vectors.items():
    block = pad_single_block(msg)
    W = schedule_from_msg(msg)
    diag_rows.append({
        "name": name,
        "len_bytes": len(msg),
        "block_hex_first_16": block[:16].hex(),
        "block_hex_last_16": block[-16:].hex(),
        "last_64_bits_as_int": int.from_bytes(block[-8:], "big"),
        "expected_bit_length": len(msg) * 8,
        "W0": fmtw(W[0]),
        "W1": fmtw(W[1]),
        "W14": fmtw(W[14]),
        "W15": fmtw(W[15]),
        "W16": fmtw(W[16]),
        "W63": fmtw(W[63]),
    })

diag_df = pd.DataFrame(diag_rows)
diag_df


,name,len_bytes,block_hex_first_16,block_hex_last_16,last_64_bits_as_int,expected_bit_length,W0,W1,W14,W15,W16,W63
0,empty,0,80000000000000000000000000000000,00000000000000000000000000000000,0,0,80000000,00000000,00000000,00000000,80000000,3b5ec49b
1,abc,3,61626380000000000000000000000000,00000000000000000000000000000018,24,24,61626380,00000000,00000000,00000018,61626380,12b1edeb
2,32_bytes_range,32,000102030405060708090a0b0c0d0e0f,00000000000000000000000000000100,256,256,00010203,04050607,00000000,00000100,4f0a6dd0,b29dedff
3,55_bytes_range,55,000102030405060708090a0b0c0d0e0f,303132333435368000000000000001b8,440,440,00010203,04050607,00000000,000001b8,732f93f7,17395ecf
4,32_bytes_ff,32,ffffffffffffffffffffffffffffffff,00000000000000000000000000000100,256,256,ffffffff,ffffffff,00000000,00000100,1ffffffe,ddff2275
5,32_bytes_zero,32,00000000000000000000000000000000,00000000000000000000000000000100,256,256,00000000,00000000,00000000,00000100,00000000,25c9488c


## 9. Controlled-subspace descent is gated off

The next phase may test:

$$
M = M_{\text{fixed}}\oplus u,\qquad u\in\{0,1\}^k.
$$

But this notebook does **not** run descent. It only makes the trace safe enough for descent.


In [11]:
print("PHASE 1151 LOCK STATUS")
print("----------------------")
print("Digest lock:", bool(lock_df["digest_match"].all()))
print("Sziklai floor lock:", bool(lock_df["sziklai_all_zero"].all()))
print("Known-preimage zero cost:", bool((cost_df["true_cost"] == 0).all()))
print()
print("Gate 1152 is now allowed: re-run ghost projection with this locked trace.")
print("Gate 1153 remains blocked until 1152 projection is measured.")


PHASE 1151 LOCK STATUS
----------------------
Digest lock: True
Sziklai floor lock: True
Known-preimage zero cost: True

Gate 1152 is now allowed: re-run ghost projection with this locked trace.
Gate 1153 remains blocked until 1152 projection is measured.


# Final Ψ-collapse

Phase 1151 locks the reference layer:

$$
\boxed{
H_{\text{engine}}=H_{\text{hashlib}}
}
$$

$$
\boxed{
C_{\text{digest}}(M_{\text{true}})=0
}
$$

$$
\boxed{
a_{r+1}-e_{r+1}\equiv T2_r-d_r
}
$$

Next gates:

$$
\boxed{
\text{Phase 1152: bitwise ghost projection re-measurement}
}
$$

then:

$$
\boxed{
\text{Phase 1153: controlled-subspace Need descent}
}
$$

Anything beyond executable SHA geometry remains tagged:

$$
\Omega
$$

until it gets a code gate.
